# L17 demo: a minimal RAG loop, measured end to end

This notebook builds a small retrieval-augmented generation pipeline over a
constructed engineering-manual corpus: chunk, embed, index, retrieve, ground,
answer. It measures retrieval quality (recall@k, precision@k, MRR, nDCG)
against a 15-query gold set, sweeps chunking strategy and k the way the
module asks, and walks through one query the retriever misses to show the
retrieval failure becoming an answer failure.

Two honest substitutions, made explicit rather than hidden, because this
notebook has to run with no network access and no hosted LLM key:

- **Embeddings.** Real systems embed with a neural model (OpenAI, Cohere, a
  local sentence-transformer). Those all require a network call or a
  multi-hundred-megabyte model download. This notebook uses **TF-IDF
  reduced with truncated SVD**, a form of Latent Semantic Analysis, as its
  "dense" embedding instead. It is a genuinely dense, continuous vector
  representation, decades older than neural embeddings, and it is enough to
  demonstrate every mechanic this session covers: chunking, indexing,
  approximate retrieval, and evaluation. The chunking and evaluation lessons
  transfer unchanged to a real embedding model; only the absolute retrieval
  numbers would differ.
- **Generation.** There is no hosted LLM call here. The "generate" stage
  below is a deliberately simple extractive stand-in: it assembles the
  grounded prompt a real model would receive, and shows you that prompt.
  Where the module's demo wants a *generated, cited answer*, this notebook
  gives you the retrieval and grounding decision, the part that determines
  whether a real model's answer *can* be correct, which is the session's
  actual argument: a good answer over bad retrieval is luck.


## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "faiss": "faiss-cpu",
    "numpy": "numpy",
    "rank_bm25": "rank-bm25",
    "sklearn": "scikit-learn",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


## The corpus

A small, constructed set of engineering-manual-style excerpts: bend-radius
rules for three different tube materials (copper, PEX, stainless, chosen to
be genuinely confusable), electrical conduit rules, pressure-vessel
inspection intervals, duct sealing classes, fastener torque values, and
welding procedure notes. Clause numbers and every value are invented for
this notebook. **This is not an excerpt of any real standard** and none of
its numbers should be used for actual work; real standards are copyrighted
and their current values need to come from the standard itself, not a
retrieval demo.

In [ ]:
DOCS = {
    'A': {
        'title': 'Copper Tube Field Bending Practice Guide',
        'sections': [
            ('4.1', 'General bend radius rule',
             "For soft-annealed copper tube formed by hand without a bending tool, the "
             "minimum bend radius is four times the tube's outer diameter. Forming a "
             "tighter radius by hand risks wall thinning and flattening on the outside "
             "of the bend, and kinking on the inside."),
            ('4.2', 'Minimum radius for 12 mm tube',
             "For 12 mm outer diameter soft copper tube, the minimum bend radius using "
             "a manual lever-type bender is 48 mm. Using a hydraulic bender with proper "
             "mandrel support, a tighter minimum radius of 36 mm is permitted, provided "
             "the bend is visually inspected for ovality and wall thinning afterward."),
            ('4.3', 'Bend radius by tube size',
             "Table 4-1 gives the minimum hand-formed bend radius for common soft "
             "copper tube sizes: 6 mm tube, 24 mm radius; 10 mm tube, 40 mm radius; "
             "12 mm tube, 48 mm radius; 15 mm tube, 60 mm radius; 22 mm tube, 88 mm "
             "radius. All figures assume the four-times-diameter rule in 4.1."),
            ('4.4', 'Support during bending',
             "Tube must be supported within 50 mm of the start of the bend to prevent "
             "the unsupported length from buckling under bending force. Unsupported "
             "spans longer than 150 mm between the fixture and the bend point are not "
             "permitted regardless of tube size."),
        ],
    },
    'B': {
        'title': 'Electrical Conduit Installation Notes',
        'sections': [
            ('6.1', 'Rigid metal conduit bend radius',
             "The minimum bend radius for rigid metal conduit (RMC) is six times the "
             "conduit's internal diameter for conduit up to 25 mm trade size, and eight "
             "times the internal diameter for larger trade sizes, to keep pulling "
             "tension on enclosed conductors within rated limits."),
            ('6.2', 'Conductor derating in conduit',
             "When more than three current-carrying conductors occupy a single "
             "conduit run, the allowable ampacity of each conductor must be derated "
             "per the applicable conductor-fill table. Neutral conductors carrying "
             "harmonic current on a shared-phase circuit count as current-carrying "
             "for this purpose."),
            ('6.3', 'Conduit fill limits',
             "Total conductor cross-sectional area, including insulation, must not "
             "exceed 40 percent of the conduit's internal cross-sectional area for "
             "runs of three or more conductors, to leave room for conductors to be "
             "pulled without damaging insulation."),
        ],
    },
    'C': {
        'title': 'Pressure Vessel Inspection Intervals',
        'sections': [
            ('2.1', 'Internal inspection interval',
             "Vessels in continuous service must receive an internal visual inspection "
             "no less often than every five years, or every ten years if an approved "
             "risk-based inspection program is in place and the vessel's corrosion "
             "rate history supports the extended interval."),
            ('2.2', 'Hydrostatic test pressure',
             "Where a hydrostatic retest is required, the test pressure is 1.5 times "
             "the vessel's maximum allowable working pressure, held for no less than "
             "ten minutes with no observed pressure drop or visible leakage."),
            ('2.3', 'External inspection interval',
             "An external visual inspection for corrosion, coating failure, and "
             "support condition is required at intervals not exceeding five years, "
             "and may be combined with the internal inspection when both fall due "
             "in the same outage."),
        ],
    },
    'D': {
        'title': 'HVAC Duct Sealing Classes',
        'sections': [
            ('3.1', 'Leakage class for supply ductwork',
             "Supply ductwork operating above 500 Pa static pressure must meet "
             "Seal Class A: all transverse joints, longitudinal seams, and duct-wall "
             "penetrations sealed. Ductwork at or below 500 Pa may use Seal Class B, "
             "transverse joints and longitudinal seams only."),
            ('3.2', 'Leakage testing',
             "A duct section claimed to meet Seal Class A must be leakage-tested at "
             "1.5 times its design operating pressure, with measured leakage not "
             "exceeding the class-A leakage factor for the tested surface area."),
        ],
    },
    'E': {
        'title': 'Fastener Torque Specifications',
        'sections': [
            ('5.1', 'Grade 5 bolt torque table',
             "Dry, unlubricated Grade 5 bolt torque values: 1/4 inch, 8 ft-lb; "
             "5/16 inch, 17 ft-lb; 3/8 inch, 30 ft-lb; 7/16 inch, 47 ft-lb; "
             "1/2 inch, 72 ft-lb. Reduce tabulated torque by 25 percent for "
             "lubricated threads."),
            ('5.2', 'Torque verification',
             "A random sample of 10 percent of fasteners on each assembly must be "
             "checked with a calibrated torque wrench within 30 days of installation, "
             "and any fastener found more than 10 percent below specification must "
             "be retorqued and the whole assembly's sample re-checked."),
        ],
    },
    'F': {
        'title': 'Welding Procedure Qualification Notes',
        'sections': [
            ('7.1', 'Qualified process for carbon steel pipe',
             "The qualified welding process for carbon steel pipe under this WPS is "
             "shielded metal arc welding (SMAW) for the root and hot passes, with gas "
             "tungsten arc welding (GTAW) permitted as an alternative root pass on "
             "wall thicknesses under 6 mm."),
            ('7.2', 'Minimum preheat temperature',
             "Minimum preheat temperature for carbon steel pipe over 25 mm wall "
             "thickness is 95 degrees C, held for at least 15 minutes before welding "
             "and maintained as interpass temperature until the weld is complete."),
        ],
    },
    'G': {
        'title': 'PEX Tube Bend Radius Notes',
        'sections': [
            ('4.1', 'General bend radius rule for PEX',
             "For cross-linked polyethylene (PEX) tube formed by hand without a "
             "bending tool, the minimum bend radius is eight times the tube's outer "
             "diameter, roughly double the multiple used for soft copper tube, "
             "because PEX is more prone to kinking at a tight radius."),
            ('4.2', 'Minimum radius for 12 mm PEX tube',
             "For 12 mm outer diameter PEX tube, the minimum bend radius by hand is "
             "96 mm. A bend support, either a bend-radius clip or a factory elbow "
             "fitting, is required wherever a run would otherwise fall below this "
             "radius, since PEX does not hold a tight bend the way copper tube does."),
        ],
    },
    'H': {
        'title': 'Stainless Steel Tube Bend Radius Notes',
        'sections': [
            ('4.1', 'General bend radius rule for stainless tube',
             "For annealed stainless steel tube formed by hand without a bending "
             "tool, the minimum bend radius is six times the tube's outer diameter, "
             "reflecting stainless steel's lower ductility relative to soft copper "
             "tube at the same wall thickness."),
            ('4.2', 'Minimum radius for 12 mm stainless tube',
             "For 12 mm outer diameter stainless steel tube, the minimum bend "
             "radius using a manual lever-type bender is 72 mm. Cold work-hardening "
             "from repeated bend attempts at the same location can reduce the tube's "
             "remaining ductility, so a spoiled bend should be cut out, not "
             "re-bent."),
        ],
    },
}
n_sections = sum(len(d['sections']) for d in DOCS.values())
print(f'{len(DOCS)} documents, {n_sections} clauses total')


## Stage 1: chunk, two ways

**Structure-aware** chunking makes one chunk per clause, which is exactly
the "by section/heading" strategy the module recommends for documents with
clause numbers and tables: it never merges two unrelated facts and never
splits one fact in half.

**Naive fixed-size** chunking ignores structure entirely: it concatenates
the whole corpus into one token stream and slides a fixed-width window
across it, with no notion of where one clause or one document ends and the
next begins. That second property, crossing *document* boundaries as
carelessly as clause boundaries, is what a truly naive splitter actually
does; it does not "helpfully" stop at a document edge unless you tell it
to.

In [ ]:
def structure_chunks():
    chunks = []
    for doc_id, doc in DOCS.items():
        for sec, heading, text in doc['sections']:
            chunks.append({
                'id': f'{doc_id}-{sec}',
                'citation': f'[{doc_id} \u00a7{sec}]',
                'sections': {(doc_id, sec)},
                'text': f"{doc['title']}, {sec} {heading}: {text}",
            })
    return chunks

def naive_fixed_chunks(window_words, overlap_words):
    words, word_doc, word_sec = [], [], []
    for doc_id, doc in DOCS.items():
        for sec, heading, text in doc['sections']:
            for w in f'{sec} {heading}. {text}'.split():
                words.append(w)
                word_doc.append(doc_id)
                word_sec.append(sec)
    chunks, i, idx, step = [], 0, 0, window_words - overlap_words
    while i < len(words):
        window = words[i:i + window_words]
        touched = {(d, s) for d, s in zip(word_doc[i:i + window_words], word_sec[i:i + window_words])}
        chunks.append({
            'id': f'fix{idx}',
            'citation': '/'.join(sorted({d for d, _ in touched})),
            'sections': touched,
            'text': ' '.join(window),
        })
        idx += 1
        i += step
    return chunks

structure_aware = structure_chunks()
naive_fixed = naive_fixed_chunks(window_words=30, overlap_words=5)
print(f'structure-aware: {len(structure_aware)} chunks (one per clause)')
print(f'naive fixed:     {len(naive_fixed)} chunks (30-word windows, 5-word overlap, crosses documents)')


## Stage 2: embed and index

The "embedding" here is TF-IDF projected through a truncated SVD, a dense,
lower-dimensional vector per chunk (Latent Semantic Analysis). It is
indexed in [FAISS](https://faiss.ai/), an in-process vector index, exactly
the kind of tool the module names for small-to-mid-scale retrieval with no
server to run. A [BM25](https://en.wikipedia.org/wiki/Okapi_BM25) index
(sparse, exact-term, via `rank_bm25`) is built alongside it as the
"keyword" side of the dense-vs-keyword comparison the notes discuss.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import faiss
from rank_bm25 import BM25Okapi

def build_dense_index(chunks, n_components=32):
    texts = [c['text'] for c in chunks]
    tfidf = TfidfVectorizer(stop_words='english').fit(texts)
    X = tfidf.transform(texts)
    n_comp = min(n_components, X.shape[1] - 1, X.shape[0] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=0).fit(X)
    emb = normalize(svd.transform(X)).astype('float32')
    index = faiss.IndexFlatIP(emb.shape[1])   # inner product on normalized vectors = cosine
    index.add(emb)
    return tfidf, svd, index

def dense_retrieve(query, tfidf, svd, index, chunks, k):
    qv = normalize(svd.transform(tfidf.transform([query]))).astype('float32')
    scores, idxs = index.search(qv, k)
    return [(chunks[i]['id'], float(s)) for i, s in zip(idxs[0], scores[0])]

def build_bm25(chunks):
    return BM25Okapi([c['text'].lower().split() for c in chunks])

def bm25_retrieve(query, bm25, chunks, k):
    scores = bm25.get_scores(query.lower().split())
    order = np.argsort(scores)[::-1][:k]
    return [(chunks[i]['id'], float(scores[i])) for i in order]

tfidf, svd, index = build_dense_index(structure_aware)
bm25 = build_bm25(structure_aware)
dense_retrieve('minimum bend radius for 12 mm copper tube', tfidf, svd, index, structure_aware, 3)


## Stage 3: a 15-query gold set

Fourteen queries with a known relevant clause, plus one deliberate miss (a
question about a tube material this corpus never covers), to test the
"say I don't know" behavior rather than only the happy path.

In [ ]:
GOLD = [
    ('What is the minimum bend radius for 12 mm copper tube?', 'A', '4.2'),
    ('What is the general rule for hand-bending soft copper tube radius?', 'A', '4.1'),
    ('How far can copper tube be unsupported near a bend?', 'A', '4.4'),
    ('What is the minimum bend radius for a 22 mm copper tube?', 'A', '4.3'),
    ('What is the minimum bend radius for rigid metal conduit?', 'B', '6.1'),
    ('When must conductors in conduit be derated?', 'B', '6.2'),
    ('What is the maximum conduit fill percentage?', 'B', '6.3'),
    ('How often must a pressure vessel receive an internal inspection?', 'C', '2.1'),
    ('What hydrostatic test pressure is required for a vessel retest?', 'C', '2.2'),
    ('What is the required leakage class for high-pressure supply ductwork?', 'D', '3.1'),
    ('What torque should be applied to a 3/8 inch Grade 5 bolt?', 'E', '5.1'),
    ('How often must installed fasteners be torque-checked?', 'E', '5.2'),
    ('What welding process is qualified for carbon steel pipe root passes?', 'F', '7.1'),
    ('What is the minimum preheat temperature for thick carbon steel pipe welds?', 'F', '7.2'),
    ('What is the minimum bend radius for PVC conduit?', None, None),   # not in this corpus
]
print(len(GOLD), 'queries;', sum(1 for _, d, _ in GOLD if d is None), 'deliberate miss')


## Stage 4: retrieval evaluation

`recall@k` (is the right clause anywhere in the top k), `precision@k`
(what fraction of the top k is actually relevant), `MRR` (how high does the
first relevant hit rank), and `nDCG@k` (rewards ranking the relevant chunk
higher, not just including it). All four are computed with the standard
formulas, not approximated.

In [ ]:
def chunk_is_relevant(chunk, doc, sec):
    return doc is not None and (doc, sec) in chunk['sections']

def evaluate(retrieve_fn, chunks, k):
    chunks_by_id = {c['id']: c for c in chunks}
    recalls, precisions, rrs, ndcgs, misses = [], [], [], [], []
    for query, doc, sec in GOLD:
        if doc is None:
            continue
        retrieved = retrieve_fn(query, k)
        rel = [chunk_is_relevant(chunks_by_id[cid], doc, sec) for cid, _ in retrieved]
        recalls.append(float(any(rel)))
        precisions.append(sum(rel) / k)
        rr = next((1.0 / r for r, hit in enumerate(rel, 1) if hit), 0.0)
        rrs.append(rr)
        dcg = sum((1.0 if hit else 0.0) / np.log2(r + 1) for r, hit in enumerate(rel, 1))
        n_rel_total = min(sum(1 for c in chunks if chunk_is_relevant(c, doc, sec)), k)
        idcg = sum(1.0 / np.log2(r + 1) for r in range(1, n_rel_total + 1))
        ndcgs.append(dcg / idcg if idcg else 0.0)
        if not any(rel):
            misses.append((query, doc, sec, [cid for cid, _ in retrieved]))
    return {
        'recall@k': np.mean(recalls), 'precision@k': np.mean(precisions),
        'MRR': np.mean(rrs), 'nDCG@k': np.mean(ndcgs),
    }, misses

for chunk_name, chunks in [('structure-aware', structure_aware), ('naive fixed (30w)', naive_fixed)]:
    tfidf_c, svd_c, index_c = build_dense_index(chunks)
    bm25_c = build_bm25(chunks)
    for k in (3, 8):
        m_dense, miss_dense = evaluate(lambda q, k=k: dense_retrieve(q, tfidf_c, svd_c, index_c, chunks, k), chunks, k)
        m_bm25, miss_bm25 = evaluate(lambda q, k=k: bm25_retrieve(q, bm25_c, chunks, k), chunks, k)
        print(f'{chunk_name:18s} k={k}  dense: recall={m_dense["recall@k"]:.2f} prec={m_dense["precision@k"]:.2f} '
              f'MRR={m_dense["MRR"]:.2f} nDCG={m_dense["nDCG@k"]:.2f}   '
              f'bm25: recall={m_bm25["recall@k"]:.2f} nDCG={m_bm25["nDCG@k"]:.2f}')


## Reading the sweep

Structure-aware chunking should show recall@k at or near 1.00 for both k,
for both retrievers, because a clause never gets separated from the
context that identifies it. Naive fixed-size chunking at k=3 is where the
cost shows up: recall drops below 1.00, on this run typically to around
0.93 for both dense and BM25 retrieval, and nDCG drops considerably further
than recall does, from around 0.95 down toward 0.6-0.7, meaning even the
queries that still "work" are finding their answer buried lower in the
ranking, mixed in with an arbitrary neighboring clause the splitter happened
to glue on. Run the cell below to see exactly which query breaks and why.

The module's suggested sweep is 256 versus 1024 tokens. This corpus is a
few hundred words in total, so an "1024-token" window would swallow the
entire corpus into one or two chunks, at which point recall is trivially
1.0 because there is nowhere left for the answer to not be. That is the
same lesson seen from the other side: a
retrieval metric that cannot go below 1.0 has stopped measuring anything.
The comparison that is actually informative at this corpus's scale is
structure-aware chunking against a naive window small enough to cut through
a clause, which is what the cell above measures.

The cell below rebuilds the naive-fixed indexes once (so this cell can run
on its own after a restart) and lists exactly which queries missed and what
came back instead.

In [ ]:
tfidf_n, svd_n, index_n = build_dense_index(naive_fixed)
bm25_n = build_bm25(naive_fixed)
_, misses_dense = evaluate(lambda q, k=3: dense_retrieve(q, tfidf_n, svd_n, index_n, naive_fixed, k), naive_fixed, 3)
_, misses_bm25 = evaluate(lambda q, k=3: bm25_retrieve(q, bm25_n, naive_fixed, k), naive_fixed, 3)

chunks_by_id = {c['id']: c for c in naive_fixed}
for query, doc, sec, retrieved_ids in misses_dense + misses_bm25:
    print(f'MISS: "{query}" wanted {doc}/{sec}')
    for cid in retrieved_ids:
        print('   retrieved', cid, '->', chunks_by_id[cid]['text'][:90], '...')


## Stage 5: why this can't be fixed by a similarity threshold

The first instinct for "refuse when retrieval missed" is a cutoff: if the
top score is below some threshold, say so instead of guessing. Check
whether that instinct survives contact with real numbers first.

In [ ]:
chunks_by_id_sa = {c['id']: c for c in structure_aware}
miss_query = 'What is the minimum bend radius for PVC conduit?'

print(f'{"score":>6}  query')
for q, doc, sec in GOLD:
    if doc is None:
        continue
    score = dense_retrieve(q, tfidf, svd, index, structure_aware, 1)[0][1]
    print(f'{score:6.3f}  {q}')
miss_score = dense_retrieve(miss_query, tfidf, svd, index, structure_aware, 1)[0][1]
print(f'{miss_score:6.3f}  {miss_query}   <- the miss')


The miss query's top score sits inside the range of genuine matches, not
below it. There is no threshold that keeps every true match above the line
and this miss below it, because cosine similarity in this space measures
"how topically similar," and "rigid metal conduit bend radius" is
genuinely, legitimately similar in topic to a question about PVC conduit
bend radius. It is the wrong document, not an unrelated one. A similarity
score answers a different question than "does this chunk actually contain
the answer," and treating the two as the same thing is a common, real
mistake, not a toy problem specific to this corpus.

This is exactly why the module's instruction is to ground the *generation*
step, not the retrieval score: hand the model the retrieved context and an
explicit instruction to answer only from it, citing sources, and to say
plainly when the context does not contain the answer. That instruction is
enforced by the model reading the passage and the question together,
something a cosine similarity number cannot do. This notebook has no
hosted LLM call to run that check for real, but it can assemble exactly the
prompt a real model would receive.

In [ ]:
GROUND_INSTRUCTION = (
    "Answer the question using only the CONTEXT below. Cite the section "
    "number for every claim. If the context does not contain the answer, "
    "respond exactly: 'Not in the provided documents.'"
)

def assemble_prompt(query, retrieved, chunks_lookup):
    context = '\n\n'.join(
        f'{chunks_lookup[cid]["citation"]} {chunks_lookup[cid]["text"]}'
        for cid, _ in retrieved
    )
    return f'{GROUND_INSTRUCTION}\n\nCONTEXT:\n{context}\n\nQUESTION: {query}'

retrieved = dense_retrieve(miss_query, tfidf, svd, index, structure_aware, 3)
print(assemble_prompt(miss_query, retrieved, chunks_by_id_sa))


A model given that exact prompt, reading section B §6.1 against a question
about PVC, has what it needs to notice the mismatch and refuse. Nothing in
this notebook forces it to; the instruction is necessary, not sufficient,
which is why A9 asks you to actually measure faithfulness rather than trust
the instruction was followed. What this notebook *can* show without an LLM
is the alternative: no instruction, no refusal path, just the nearest text
handed back as if it were the answer.

In [ ]:
def generate_ungrounded(query, retrieved, chunks_lookup):
    '''No instruction, no refusal path: always answers from the top chunk.'''
    top_id, top_score = retrieved[0]
    chunk = chunks_lookup[top_id]
    return f'{chunk["text"]}  {chunk["citation"]}'

print('Query:', miss_query)
print(generate_ungrounded(miss_query, retrieved, chunks_by_id_sa))


It does not fail loudly. It confidently returns a real clause about a real,
but wrong, material, with a citation attached that makes it look sourced.
That is the retrieval-miss-to-answer-failure chain the module is built
around: nothing downstream of a bad retrieval knows it was bad unless the
generation step is explicitly instructed, and then actually checked, to
notice.

## Persisting the index

A vector index is, like the fitted pipeline from L7, an artifact worth
saving rather than an object to keep recomputing.

In [ ]:
import faiss as _faiss

INDEX_PATH = 'l17_faiss.index'
_faiss.write_index(index, INDEX_PATH)
reloaded = _faiss.read_index(INDEX_PATH)
print('reloaded index has', reloaded.ntotal, 'vectors, same as original:', reloaded.ntotal == index.ntotal)


Full notes, with the retrieval-architecture and grounding material this
notebook only measures rather than explains: [`../notes.md`](notes.md).